In [24]:
%pip install pandas
%pip install matplotlib
%pip install statsmodels

import pandas as pd

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
data = pd.read_csv("../data/freehire_tech.csv")

Data Cleaning Log
1. Keep jobs in US.
2. Keep jobs in 2026.
3. Generate salary_mid_year column.
    Filter out job listings having nulls in salary columns (salary_min, salary_max, salary_currency, salary_period) and not in USD currency.
    Convert all salary to year salary, and then calculate salary_mid_year = (salary_max_year + salary_min_year) / 2 
    
4. Generate State column based on location. There's another column called location_type (i.e. state, us_only, remote, untraditional, missing), which used to note down those listings which location is hard to put down to a state.
5. Divide skills into categories. ("Manual" keyword search. Need to try Embedding.)
6. Nulls in most columns haven't been processed yet. To be done.

In [26]:
import ast

#us_jobs = data[data["countries"].apply(lambda x: "us" in ast.literal_eval(x))].copy()


us_jobs = data[data["countries"].apply(lambda x: ast.literal_eval(x) == ["us"])].copy()

In [27]:
us_jobs["posted_at"] = pd.to_datetime(us_jobs["posted_at"], format="ISO8601", utc=True)

us_jobs["posted_at"].dt.year.value_counts().sort_index()

posted_at
1970.0         1
2012.0        51
2013.0       210
2014.0       422
2015.0      2026
2016.0      3609
2017.0      3940
2018.0       180
2019.0       276
2020.0       240
2021.0       400
2022.0       552
2023.0      1130
2024.0      5619
2025.0     11745
2026.0    311210
Name: count, dtype: int64

In [28]:
us_jobs_2026 = us_jobs[us_jobs["posted_at"].dt.year == 2026]

us_jobs_2026.shape

(311210, 27)

In [29]:
salary_cols = ["salary_min", "salary_max", "salary_currency", "salary_period"]
predictor_cols = ["title", "work_mode", "skills", "seniority", "category", "employment_type",
                  "experience_years_min", "company_industries", "company_types", "company_sizes"]

print("Drop on salary only:", us_jobs_2026.dropna(subset=salary_cols).shape[0])
print("Drop on everything: ", us_jobs_2026.dropna(subset=salary_cols + predictor_cols).shape[0])

Drop on salary only: 82817
Drop on everything:  5413


In [30]:
us_jobs_2026_withsalary = us_jobs_2026.dropna(subset=salary_cols).copy()

print(us_jobs_2026_withsalary["salary_currency"].value_counts())


salary_currency
USD    82092
usd      587
CAD       62
EUR       32
AUD        7
GBP        6
PLN        6
INR        5
ZAR        5
PHP        4
MXN        3
COP        2
HKD        2
eur        1
nzd        1
clp        1
BRL        1
Name: count, dtype: int64


In [31]:
us_jobs_2026_withsalary["salary_currency"] = us_jobs_2026_withsalary["salary_currency"].str.upper().str.strip()

us_jobs_2026_withsalary = us_jobs_2026_withsalary[us_jobs_2026_withsalary["salary_currency"] == "USD"].copy()

print(len(us_jobs_2026_withsalary))

82679


In [32]:
# standard full-time conversions: 2,080 hours (40 × 52), 260 working days (5 × 52), and 12 months per year.
multiplier = {"year": 1, "month": 12, "day": 260, "hour": 2080}

factor = us_jobs_2026_withsalary["salary_period"].map(multiplier)

us_jobs_2026_withsalary["salary_min_year"] = us_jobs_2026_withsalary["salary_min"] * factor
us_jobs_2026_withsalary["salary_max_year"] = us_jobs_2026_withsalary["salary_max"] * factor
us_jobs_2026_withsalary["salary_mid_year"] = (us_jobs_2026_withsalary["salary_min_year"] + us_jobs_2026_withsalary["salary_max_year"]) / 2

print(us_jobs_2026_withsalary[["salary_min_year", "salary_max_year", "salary_mid_year"]].describe())

       salary_min_year  salary_max_year  salary_mid_year
count     8.267900e+04     8.267900e+04     8.267900e+04
mean      1.666193e+05     2.381658e+05     2.023925e+05
std       2.029601e+06     2.635362e+06     2.266991e+06
min       1.000000e+00     1.000000e+00     1.000000e+00
25%       8.373600e+04     1.200000e+05     1.033250e+05
50%       1.175000e+05     1.700000e+05     1.450000e+05
75%       1.530000e+05     2.215000e+05     1.875000e+05
max       4.142486e+08     4.142486e+08     4.142486e+08


In [33]:
%pip install geonamescache us
import re
import pandas as pd
import us
import geonamescache

# ---- Settings ----
df = us_jobs_2026_withsalary
LOC_COL = "location"
OTHER_LABEL = "untraditional"    # label for locations that name a site, not a place
EXTRA_CITIES = {
    "nyc": "NY", "bay area": "CA", "silicon valley": "CA",
    "dmv": "DC", "research triangle": "NC",
}

# ---- Lookups ----
STATE_NAMES = {s.name.lower(): s.abbr for s in us.states.STATES + [us.states.DC]}
STATE_ABBRS = set(STATE_NAMES.values())

# US cities with population > 15k; if a name repeats, the larger city wins
gc = geonamescache.GeonamesCache()
us_cities = sorted(
    (c for c in gc.get_cities().values() if c["countrycode"] == "US"),
    key=lambda c: c["population"],
)
CITY_TO_STATE = {c["name"].lower(): c["admin1code"] for c in us_cities}
CITY_TO_STATE.update(EXTRA_CITIES)

# longest names first so "West Virginia" matches before "Virginia"
STATE_NAME_RE = re.compile(
    r"\b(" + "|".join(sorted(map(re.escape, STATE_NAMES), key=len, reverse=True)) + r")\b", re.I
)
ABBR_RE    = re.compile(r"(?<![A-Za-z])([A-Z]{2})(?![A-Za-z])")
REMOTE_RE  = re.compile(r"\b(remote|anywhere|work from home|wfh)\b", re.I)
NATION_RE  = re.compile(r"\b(nationwide|multiple locations|various locations)\b", re.I)
COUNTRY_RE = re.compile(r"\b(united states( of america)?|usa|u\.s\.a?\.?|us)\b", re.I)
AREA_RE    = re.compile(r"^greater\s+|\s+(bay|metro(politan)?)?\s*area$", re.I)

# ---- Helpers ----
def split_parts(loc):
    """'Remote - US' -> ['Remote', 'US']"""
    loc = re.sub(r"\s*[-–—|/;()]\s*", ",", loc)
    return [p.strip() for p in loc.split(",") if p.strip()]

def find_state(loc):
    parts = split_parts(loc)

    # 1. a whole part is a state abbreviation or name ("Seattle, WA")
    for p in reversed(parts):
        if p in STATE_ABBRS:
            return p
        if p.lower() in STATE_NAMES:
            return STATE_NAMES[p.lower()]

    # 2. a whole part is a known city or metro area ("San Francisco")
    for p in parts:
        for key in (p.lower(), AREA_RE.sub("", p).lower()):
            if key in CITY_TO_STATE:
                return CITY_TO_STATE[key]

    # 3. state abbreviation or name inside the text ("Atlanta GA", "USA.VA.Reston")
    abbrs = [a for a in ABBR_RE.findall(loc) if a in STATE_ABBRS]
    if abbrs:
        return abbrs[-1]
    m = STATE_NAME_RE.search(loc)
    if m:
        return STATE_NAMES[m.group(1).lower()]

    # 4. city name inside the text ("Denver HQ"), longest phrase first
    words = re.findall(r"[a-z]+", loc.lower())
    for n in (3, 2, 1):
        for i in range(len(words) - n + 1):
            phrase = " ".join(words[i:i + n])
            if phrase in CITY_TO_STATE:
                return CITY_TO_STATE[phrase]
    return None

def location_type(loc, state):
    if pd.notna(state):
        return "state"
    if not isinstance(loc, str) or not loc.strip():
        return "missing"
    if REMOTE_RE.search(loc):
        return "remote"
    if NATION_RE.search(loc) or not re.sub(r"[\W_]+", "", COUNTRY_RE.sub("", loc)):
        return "us_only"
    return OTHER_LABEL

# ---- Apply (once per unique location, then map back) ----
unique_locs = df[LOC_COL].dropna().unique()
state_lookup = {loc: find_state(loc) for loc in unique_locs}

df["state"] = df[LOC_COL].map(state_lookup)
df["location_type"] = [location_type(l, s) for l, s in zip(df[LOC_COL], df["state"])]

# ---- Check ----
print(df["location_type"].value_counts(normalize=True).mul(100).round(1))
df.loc[df["location_type"] == OTHER_LABEL, LOC_COL].value_counts().head(30)

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
location_type
state            83.3
us_only          11.0
remote            5.2
untraditional     0.4
missing           0.0
Name: proportion, dtype: float64


location
PICKLE RESEARCH CAMPUS                         24
International                                  14
Hybrid - Local to D.C. metro area              11
Any location, United States                    10
JPL Campus                                     10
United States - Field                          10
Okinawa                                         8
USA wide, us                                    7
Evendale                                        6
Unity One                                       6
Greenwich, United States                        5
United States - Flex                            5
RWC HQ                                          5
Ramstein-Miesenbach, Rhineland-Palatinate       4
ALHU12                                          4
Jessup, United States                           4
Northeast - United States                       4
Continental United States                       4
VACH06                                          4
Napoli                                   

In [34]:
df.head().T

,1,2,5,6,10
public_slug,lead-software-engineer-automation-2k-6e4grigf,lead-software-engineer-automation-2k-kasazizl,senior-software-engineer-automation-2k-7pii2rzx,senior-software-engineer-automation-2k-er2tqhp2,operations-program-manager-supply-chain-10beau...
source,greenhouse,greenhouse,greenhouse,greenhouse,greenhouse
title,Lead Software Engineer - Automation,Lead Software Engineer - Automation,Senior Software Engineer - Automation,Senior Software Engineer - Automation,Operations Program Manager (Supply Chain)
company,2K,2K,2K,2K,10Beauty
company_slug,2k,2k,2k,2k,10beauty
location,"Culver City, California, United States","Novato, California, United States","Culver City, California, United States","Novato, California, United States","Burlington, Massachusetts, United States"
url,https://job-boards.greenhouse.io/2kvegas/jobs/...,https://job-boards.greenhouse.io/2kvegas/jobs/...,https://job-boards.greenhouse.io/2kvegas/jobs/...,https://job-boards.greenhouse.io/2kvegas/jobs/...,https://job-boards.greenhouse.io/10beauty/jobs...
posted_at,2026-06-17 04:24:24+00:00,2026-06-17 04:24:25+00:00,2026-06-11 16:45:29+00:00,2026-06-11 16:45:29+00:00,2026-07-24 16:52:37+00:00
countries,['us'],['us'],['us'],['us'],['us']
regions,['north_america'],['north_america'],['north_america'],['north_america'],['north_america']


In [35]:
state_salary = (df
                .groupby("state")["salary_mid_year"]
                .agg(["median", "count"])
                .query("count >= 100")
                .sort_values("median", ascending=False))

state_salary

,median,count
state,,
CA,179000.00,18128
WA,174225.50,4136
NY,161500.00,6833
DC,145000.00,1304
VA,142300.00,4410
MD,140025.00,2260
MA,137500.00,3845
TX,135000.00,3492
CO,129500.00,2241


In [36]:
import ast

df["skills_list"] = df["skills"].apply(ast.literal_eval)

skill_counts = df["skills_list"].explode().value_counts()

print("Unique skills:", len(skill_counts))
print("Jobs with no skills listed:", (df["skills_list"].str.len() == 0).sum())
print("Share of mentions covered by top 150:", round(skill_counts.head(150).sum() / skill_counts.sum() * 100, 1), "%")

print(", ".join(skill_counts.head(150).index))

Unique skills: 729
Jobs with no skills listed: 14031
Share of mentions covered by top 150: 78.4 %
ai, cloud, python, automation, analytics, api, aws, machine-learning, agile, sql, ci-cd, azure, networking, typescript, kubernetes, java, gcp, agentic-ai, observability, cpp, llm, linux, data-science, devops, distributed-systems, cybersecurity, docker, javascript, program-management, saas, terraform, infrastructure-as-code, generative-ai, react, data-pipelines, cloud-native, git, jira, github, microservices, data-engineering, accessibility, test-automation, bash, scrum, data-quality, statistics, rest, postgresql, stakeholder-management, spark, csharp, data-analytics, version-control, kafka, prototyping, process-improvement, oracle, powerbi, containerization, data-modeling, salesforce, snowflake, sap, authentication, tableau, databricks, iam, jenkins, etl, data-governance, event-driven-architecture, devsecops, go-to-market, design-patterns, sdlc, gitlab, erp, android, nodejs, ios, crm, nosq

In [37]:
import numpy as np
import statsmodels.formula.api as smf

# ---- Option B: AI-focused skill categories ----
skill_categories = {
    "genai": [
        "llm", "generative-ai", "agentic-ai", "prompt-engineering", "openai", "claude-code"
    ],
    "traditional_ml": [
        "machine-learning", "deep-learning", "pytorch", "tensorflow", "computer-vision",
        "nlp", "mlops", "sensor-fusion"
    ],
    "data": [
        "analytics", "sql", "data-science", "data-pipelines", "data-engineering", "data-quality",
        "statistics", "postgresql", "spark", "data-analytics", "kafka", "oracle", "powerbi",
        "data-modeling", "snowflake", "tableau", "databricks", "etl", "data-governance", "nosql",
        "data-visualization", "data-warehousing", "sql-server", "mysql", "airflow", "matlab"
    ],
    "infrastructure": [
        # cloud & devops
        "cloud", "automation", "aws", "ci-cd", "azure", "networking", "kubernetes", "gcp",
        "observability", "linux", "devops", "docker", "terraform", "infrastructure-as-code",
        "cloud-native", "containerization", "jenkins", "gitlab", "ansible", "github-actions",
        "cisco", "powershell", "bash", "servicenow", "grafana", "dns", "azure-devops",
        "virtualization", "distributed-computing", "tcp-ip", "backbone", "datadog", "unix",
        "prometheus", "lambda",
        # security
        "cybersecurity", "authentication", "iam", "devsecops", "cloud-security", "firewall",
        "regulatory-compliance", "network-security", "zero-trust", "siem", "splunk"
    ],
    "software_eng": [
        "python", "api", "typescript", "java", "cpp", "distributed-systems", "javascript",
        "react", "git", "github", "microservices", "test-automation", "rest", "csharp",
        "version-control", "event-driven-architecture", "design-patterns", "sdlc", "android",
        "nodejs", "ios", "html", "css", "spring", "rust", "oop", "scala", "dotnet", "angular",
        "embedded-systems", "developer-experience", "api-design", "kotlin", "embedded-software",
        "express", "graphql", "assembly"
    ]
}

#df = us_jobs_2026_withsalary

# ---- Create dummies ----
for cat, skills in skill_categories.items():
    df[f"skill_{cat}"] = df["skills_list"].apply(lambda lst: int(any(s in skills for s in lst)))

df["skill_ai_generic"] = df["skills_list"].apply(lambda lst: int("ai" in lst))   # broad "ai" tag, kept separate
df["skills_listed"] = (df["skills_list"].str.len() > 0).astype(int)

skill_cols = [f"skill_{c}" for c in skill_categories] + ["skill_ai_generic"]

# ---- Quick checks ----
print("Share of jobs (that list skills) in each category:")
print(df.loc[df["skills_listed"] == 1, skill_cols].mean().round(3), "\n")

print("Median salary without (0) vs with (1) each category:")
for c in skill_cols:
    print(c, df.groupby(c)["salary_mid_year"].median().to_dict())

print("\nGenAI vs traditional ML overlap:")
print(pd.crosstab(df["skill_genai"], df["skill_traditional_ml"]))

Share of jobs (that list skills) in each category:
skill_genai             0.207
skill_traditional_ml    0.220
skill_data              0.469
skill_infrastructure    0.680
skill_software_eng      0.616
skill_ai_generic        0.514
dtype: float64 

Median salary without (0) vs with (1) each category:
skill_genai {0: 135250.0, 1: 182500.0}
skill_traditional_ml {0: 135000.0, 1: 185723.5}
skill_data {0: 135000.0, 1: 157500.0}
skill_infrastructure {0: 113450.0, 1: 163000.0}
skill_software_eng {0: 117700.0, 1: 166800.0}
skill_ai_generic {0: 124000.0, 1: 170000.0}

GenAI vs traditional ML overlap:
skill_traditional_ml      0     1
skill_genai                      
0                     60102  8340
1                      7467  6770


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 82679 entries, 1 to 876676
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   public_slug           82679 non-null  object             
 1   source                82679 non-null  object             
 2   title                 82679 non-null  object             
 3   company               81520 non-null  object             
 4   company_slug          81520 non-null  object             
 5   location              82655 non-null  object             
 6   url                   82679 non-null  object             
 7   posted_at             82679 non-null  datetime64[ns, UTC]
 8   countries             82679 non-null  object             
 9   regions               82679 non-null  object             
 10  work_mode             34127 non-null  object             
 11  skills                82679 non-null  object             
 12  cities  